# Microglia Somal Segmentation using CellPose-SAM

## Set constants

In [10]:
# Test variables
USE_GPU = False
PROP_SLICES = 0.05  # Low for testing, higher(/all) for training

## Load annotated data

#### Installations and imports

In [2]:
!pip install -q scikit-learn tifffile tensorflow stardist csbdeep scikit-image imagecodecs
print("✓ Installations complete.")

✓ Installations complete.


In [3]:
try:
    import os
    if not USE_GPU:
        os.environ["CUDA_VISIBLE_DEVICES"] = ""
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # Limit warning messages
    from csbdeep.utils import normalize
    import numpy as np
    from math import ceil
    import tifffile as tiff
    from tifffile import imread, imwrite
    from skimage.measure import regionprops
    from skimage.segmentation import relabel_sequential
    from scipy import ndimage as ndi
    from sklearn.model_selection import train_test_split
    from pathlib import Path
    print("✓ Imported packages.")
except:
    print("× Package imports failed.")

✓ Imported packages.


#### Set directory constants

In [4]:
DATA_DIR = Path("data")
PROJECT_DIR = Path.cwd()
IMG_DIR = DATA_DIR / "images"
LBL_DIR = DATA_DIR / "labels"
IMG_SLICE_DIR = IMG_DIR / "slices"
LBL_SLICE_DIR = LBL_DIR / "slices"
MODEL_DIR = PROJECT_DIR / "models"
PRED_DIR = DATA_DIR / "predictions"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Image Directory: {IMG_SLICE_DIR}")

Image Directory: data/images/slices


#### Slice each .tiff along certain axes (for CellPose)

In [5]:
def iterative_image_splice(image_dir, label_dir, output_image_dir, output_label_dir, p=1):
    if p <= 0 or p > 1:
        raise ValueError("p must be in (0, 1]")
    else:
        q = round(1 / p)

    print(f"✓ Using q = {q}\n")
    
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)
    
    tif_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.tif')])
    
    for tif_file in tif_files:
        if tif_file.startswith('.'):
            continue
        
        image_path = os.path.join(image_dir, tif_file)
        label_path  = os.path.join(label_dir, tif_file)
        
        if not os.path.exists(label_path):
            print(f"Skipping {tif_file}: no matching label.")
            continue
        
        # Load volumes
        with tiff.TiffFile(image_path) as tif:
            image = tif.asarray()
        with tiff.TiffFile(label_path) as tif:
            label = tif.asarray()
        
        # Check alignment
        if image.shape != label.shape:
            raise ValueError(f"Shape mismatch for {tif_file}: {image.shape} vs {label.shape}")
        
        z, y, x = image.shape
        base_name = os.path.splitext(tif_file)[0]
        
        # --- XY slices ---
        for z_coord in range(0, z, q):
            img_xy = image[z_coord, :, :]
            msk_xy = label[z_coord, :, :]
            
            tiff.imwrite(os.path.join(output_image_dir, f'{base_name}_XY_Z{z_coord}.tif'), img_xy)
            tiff.imwrite(os.path.join(output_label_dir, f'{base_name}_XY_Z{z_coord}.tif'), msk_xy)
        
        # --- XZ slices ---
        for x_coord in range(0, x, q):
            img_xz = image[:, :, x_coord]
            msk_xz = label[:, :, x_coord]
            
            tiff.imwrite(os.path.join(output_image_dir, f'{base_name}_XZ_Y{x_coord}.tif'), img_xz)
            tiff.imwrite(os.path.join(output_label_dir, f'{base_name}_XZ_Y{x_coord}.tif'), msk_xz)

        # --- YZ slices ---
        for y_coord in range(0, y, q):
            img_yz = image[:, y_coord, :]
            msk_yz = label[:, y_coord, :]
            
            tiff.imwrite(os.path.join(output_image_dir, f'{base_name}_YZ_X{y_coord}.tif'), img_yz)
            tiff.imwrite(os.path.join(output_label_dir, f'{base_name}_YZ_X{y_coord}.tif'), msk_yz)

        print(f"✓ Sliced: {tif_file}")

iterative_image_splice(IMG_DIR, LBL_DIR, IMG_SLICE_DIR, LBL_SLICE_DIR, p=PROP_SLICES)

✓ Using q = 20

✓ Sliced: image1-Copy1.tif
✓ Sliced: image1-Copy2.tif
✓ Sliced: image1.tif


#### Read, normalize, and split slice data

In [6]:
# Data processing functions (pad, load pair) and initial data load

def pad_to_multiple(arr, multiple=16, constant_value=0):
    spatial = arr.shape[1:3]   # ONLY Y, X
    target = [int(ceil(s / multiple) * multiple) for s in spatial]
    pad_width = [(0, 0)]  # Z unchanged
    pad_width += [(0, t - s) for s, t in zip(spatial, target)]
    if arr.ndim == 4:
        pad_width.append((0, 0))
    return np.pad(arr, pad_width, mode="constant", constant_values=constant_value)

def load_pair(sample_id):
    """Loads the image and label associated with the given filename."""
    img = imread(IMG_SLICE_DIR / f"{sample_id}.tif").astype(np.float32)   # expected ZYX
    lbl = imread(IMG_SLICE_DIR / f"{sample_id}.tif").astype(np.int32)     # expected ZYX

    lbl, _, _ = relabel_sequential(lbl)

    assert img.shape == lbl.shape, f"Shape mismatch for {sample_id}: {img.shape} vs {lbl.shape}"
    assert lbl.dtype.kind in ("i", "u"), f"Label dtype must be integer for {sample_id}"

    # Normalize the image to control for differing grayscale ranges
    img = normalize(img, 1, 99.8, axis=(0,1))

    img = pad_to_multiple(img, multiple=16, constant_value=0)
    lbl = pad_to_multiple(lbl, multiple=16, constant_value=0)

    return img, lbl

def generate_XY(sample_ids):
    X, Y = [], []
    for sample_id in sample_ids:
        img, lbl = load_pair(sample_id)
        X.append(img)
        Y.append(lbl)
        
    # X = [np.stack([img], axis=0) for img in X]  # transform into (1, H, W)
    
    return X, Y

sample_ids = sorted([p.stem for p in IMG_SLICE_DIR.glob("*.tif")])
X, Y = generate_XY(sample_ids)
print(f"Loaded {len(sample_ids)} image/label pairs.")

print(f"\n{'#':<4} {'Filename ID':<20} {'Shape':<18}")
print("-" * 47)
for i, id in enumerate(sample_ids):
    print(f"{i+1:<4} {id:<20} {str(X[i].shape):<18}")

Loaded 336 image/label pairs.

#    Filename ID          Shape             
-----------------------------------------------
1    image1-Copy1_XY_Z0   (664, 752)        
2    image1-Copy1_XY_Z1   (664, 752)        
3    image1-Copy1_XY_Z10  (664, 752)        
4    image1-Copy1_XY_Z11  (664, 752)        
5    image1-Copy1_XY_Z12  (664, 752)        
6    image1-Copy1_XY_Z13  (664, 752)        
7    image1-Copy1_XY_Z14  (664, 752)        
8    image1-Copy1_XY_Z15  (664, 752)        
9    image1-Copy1_XY_Z16  (664, 752)        
10   image1-Copy1_XY_Z17  (664, 752)        
11   image1-Copy1_XY_Z18  (664, 752)        
12   image1-Copy1_XY_Z19  (664, 752)        
13   image1-Copy1_XY_Z2   (664, 752)        
14   image1-Copy1_XY_Z20  (664, 752)        
15   image1-Copy1_XY_Z21  (664, 752)        
16   image1-Copy1_XY_Z22  (664, 752)        
17   image1-Copy1_XY_Z23  (664, 752)        
18   image1-Copy1_XY_Z24  (664, 752)        
19   image1-Copy1_XY_Z25  (664, 752)        
20   image1-Copy1_XY_

In [7]:
# Normalization and train/validate/test split

def norm(X):
    if X is None:
        return None
    out = []
    for x in X:
        x_norm = normalize(x, 1, 99.8, axis=(0,1))
        out.append(x_norm)
    return out

print("Normalizing and splitting data...", end = " ")
n = len(X)

if n < 3:
    raise ValueError("Sample size < 3")
elif n < 6:
    # Only train/test sets
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3)
    X_val, Y_val = None, None
else:
    # Full train/validate/test split
    X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.3)
    X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5)

X_train_norm = norm(X_train)

if X_val is not None: X_val_norm = norm(X_val)
else: X_val_norm = None

X_test_norm = norm(X_test)

print(f"finished, no issues.")

assert type(X_train_norm) == type(Y_train), f"X_train_norm type ({type(X_train_norm)}) does not match Y_train type ({type(Y_train)})"
print(f"X_train_norm/Y_train type: {type(X_train_norm)}")
print(f"X_train_norm[0] shape: {X_train_norm[0].shape}")
print(f"Y_train[0] shape: {Y_train[0].shape}")

Normalizing and splitting data... finished, no issues.
X_train_norm/Y_train type: <class 'list'>
X_train_norm[0] shape: (41, 752)
Y_train[0] shape: (41, 752)


## Load and finetune model

#### Installs and Imports

In [8]:
!pip uninstall -y cellpose
!pip cache purge
!pip install -q cellpose==4.0.1
from cellpose import models, train, plot, core, io
print("✓ Installations complete.")

Found existing installation: cellpose 3.1.1.2
Uninstalling cellpose-3.1.1.2:
  Successfully uninstalled cellpose-3.1.1.2
Files removed: 6 (278 kB)
Directories removed: 0
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Welcome to CellposeSAM, cellpose v4.0.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


✓ Installations complete.


#### Load and train model

In [11]:
# Load model
model = models.CellposeModel(gpu=USE_GPU)
print(f"✓ Loaded pretrained model; using {'GPU' if USE_GPU else 'CPU'}.")
print("CellPose Architecture: " + str(type(model.net)))

✓ Loaded pretrained model; using CPU.
CellPose Architecture: <class 'cellpose.vit_sam.Transformer'>


In [12]:
# Training hyperparameters
n_epochs = 100
learning_rate = 0.1
weight_decay = 0.0001
batch_size = 8

# Channel configuration: [cytoplasm_channel, nucleus_channel]
# For grayscale: [0, 0]
channels = [0, 0]  # Grayscale
channel_axis = 0 if X_train_norm[0].ndim == 3 else None

print(f"✓ Hyperparameters set.")
print(f"    ✓ channel_axis = {channel_axis}")

# Model save configuration
model_name = 'cellpose_sam_finetuned'

print(f"✓ Model directory located.")

✓ Hyperparameters set.
    ✓ channel_axis = None
✓ Model directory located.


In [13]:
def check_all_nd(arr_list):
    n = arr_list[0].ndim
    
    bad_indices = []

    for i, arr in enumerate(arr_list):
        if not isinstance(arr, np.ndarray):
            print(f"Index {i}: not a numpy array (type={type(arr)})")
            bad_indices.append(i)
            continue

        if arr.ndim != n:
            print(f"Index {i}: has shape {arr.shape}, ndim={arr.ndim}")
            bad_indices.append(i)

    if len(bad_indices) == 0:
        print(f"✓ All arrays are {n}D")
    else:
        print(f"× {len(bad_indices)} arrays are not 3D")

check_all_nd(X_train_norm)
check_all_nd(Y_train)

✓ All arrays are 2D
✓ All arrays are 2D


In [14]:
print("Starting training...\n")
print(f"Shape of each X: {X_train_norm[0].shape}")
print(f"Shape of each Y: {Y_train[0].shape}")

model_path = train.train_seg(
    model.net,
    train_data=X_train_norm,
    train_labels=Y_train,
    test_data=X_val_norm if X_val_norm is not None else None,
    test_labels=Y_val if Y_val is not None else None,
    save_path=MODEL_DIR,
    save_every=10,
    n_epochs=n_epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    batch_size=batch_size,
    model_name=model_name,
    min_train_masks=1,  # Threshold of unique values per training image/label pair
    rescale=True,  # Rescale images based on diameter
    channel_axis=channel_axis,
)

print(f"\n✓ Training complete!")
print(f"✓ Model saved to: {model_path}")

# "GOT OUT OF MEMORY ERROR"

Starting training...

Shape of each X: (41, 752)
Shape of each Y: (41, 752)


  3%|▎         | 7/235 [00:09<05:00,  1.32s/it]


KeyboardInterrupt: 

## Predict and evaluate

In [ ]:
# Loss functions
def dice_score(y_true, y_pred):
    y_true = (y_true > 0).astype(np.bool_)
    y_pred = (y_pred > 0).astype(np.bool_)
    intersection = np.logical_and(y_true, y_pred).sum()
    return 2. * intersection / (y_true.sum() + y_pred.sum() + 1e-8)

def iou_score(y_true, y_pred):
    y_true = (y_true > 0).astype(np.bool_)
    y_pred = (y_pred > 0).astype(np.bool_)
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return intersection / (union + 1e-8)

preds = []

for x in X:
    pred = model.eval(x)[0]  # common Cellpose pattern
    preds.append(pred)

print(f"✓ Predicted {len(preds)} samples")

dice_scores = []
iou_scores = []

for y_true, y_pred in zip(Y, preds):
    dice_scores.append(dice_score(y_true, y_pred))
    iou_scores.append(iou_score(y_true, y_pred))

print("\n===== RESULTS =====")
print(f"Mean Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"Mean IoU : {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")